In [6]:
import os
import pandas as pd
from openai import OpenAI
import time
import json
import re
from typing import Any, Dict, List

# Setup network bypass for local infrastructure
os.environ['no_proxy'] = '10.0.0.27,localhost,127.0.0.1'

# =========================
# 1) Configuration
# =========================
INPUT_XLSX_PATH = "/Users/strangemax/syncthing/cirfolder/DATASCIENCE FOR HEALTH/Semester 2 /DASC512-202526/Assignment2/git12/512_2/205_predicted_normal_pool.xlsx"
OUTPUT_XLSX_PATH = "/Users/strangemax/syncthing/cirfolder/DATASCIENCE FOR HEALTH/Semester 2 /DASC512-202526/Assignment2/git12/512_2/LMoutput/205_normal_bullets_qwen3-30b-a3b-2507_combined_testdata1.xlsx"
BATCH_SIZE = 1  # Kept at 1 for row-by-row file writing and resuming
SHEET_NAME = 0  

# LM Studio Configuration
MODEL_NAME = "qwen3-30b-a3b-2507"
client = OpenAI(
    base_url="http://localhost:1234/v1", 
    api_key="lm-studio", 
    timeout=600.0  
)

# =========================
# 2) Helpers
# =========================
def extract_json(text: str) -> Dict[str, Any]:
    """Pulls out the first JSON object from the model output."""
    text = (text or "").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Single-line clean regex pattern to avoid multi-line string termination issues
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        return json.loads(fence.group(1))

    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in model output.")
    return json.loads(m.group(1))

def normalize_findings(findings: Any) -> List[str]:
    """Cleans, strips bullet points, and deduplicates findings."""
    if not isinstance(findings, list):
        return []

    cleaned: List[str] = []
    for f in findings:
        s = str(f).strip()
        if not s:
            continue
        s = re.sub(r"^[-•\*\u2022]\s*", "", s).strip()
        if s:
            cleaned.append(s)

    seen = set()
    out: List[str] = []
    for s in cleaned:
        key = s.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)

    return out

# =========================
# 3) Load Data & Manage Resume State
# =========================
df_source = pd.read_excel(INPUT_XLSX_PATH, sheet_name=SHEET_NAME, engine="openpyxl")
df_source = df_source.reset_index(drop=True)

# Add standard index tracker to source data so we can map it back safely
df_source['original_row_index'] = df_source.index

# Check if an output file exists to resume progress
if os.path.exists(OUTPUT_XLSX_PATH):
    print(f"Found existing output file: {OUTPUT_XLSX_PATH}. Checking progress...")
    df_existing = pd.read_excel(OUTPUT_XLSX_PATH, engine="openpyxl")
    
    if not df_existing.empty and 'original_row_index' in df_existing.columns:
        last_processed_idx = int(df_existing['original_row_index'].max())
        start_row = last_processed_idx + 1
        print(f"Resuming pipeline from row index {start_row} (Processed up to index {last_processed_idx}).")
    else:
        print("Output file structure unreadable or empty. Starting from scratch.")
        df_existing = pd.DataFrame(columns=df_source.columns.tolist() + ["is_abnormal", "findings"])
        start_row = 0
else:
    print("No existing output file found. Initializing a new run.")
    df_existing = pd.DataFrame(columns=df_source.columns.tolist() + ["is_abnormal", "findings"])
    start_row = 0

print(f"Total reports to process: {len(df_source)}. Remaining: {len(df_source) - start_row}")

# =========================
# 4) Processing Loop
# =========================
for start in range(start_row, len(df_source), BATCH_SIZE):
    batch_df = df_source.iloc[start: start + BATCH_SIZE]
    
    # Use the original row index explicitly for LLM data binding
    batch_data = batch_df[["original_row_index", "text"]].to_csv(index=False)

    prompt = f"""
You will be given multiple radiology reports.

TASK (for EACH report):
Identify if there is ANY abnormal finding present. This includes:
- New abnormal findings.
- Abnormal findings previously referenced in the past.
- Findings that are worsening.
- Findings that are remaining stable/the same.

Extract these abnormal findings as a list of bullet points.

Return ONLY valid JSON in this exact structure:
{{
  "results": [
    {{
      "original_row_index": 0,
      "findings": [
        "finding 1",
        "finding 2"
      ]
    }}
  ]
}}

INCLUSION RULES:
- Include ANY abnormal finding that is confirmed OR even slightly suspected (e.g., "possible", "may represent", "cannot exclude", "concerning for", "question of").
- Include findings whether they are new, chronic, stable, improving, or worsening.
- Include past abnormalities mentioned in the history/comparison text unless they are noted as completely resolved now.
- Ignore magnitude/certainty modifiers (small/slight/minimal/tiny) — still include the finding.

EXCLUSION RULES:
- Do NOT include findings that are completely NEGATED/ABSENT (e.g., "no pneumothorax", "without effusion", "negative for consolidation", "resolved").
- Do NOT include items that are merely listed as differential examples unless the report actually suggests them as present/suspected in this patient.
- Do NOT include man-made post-surgical devices/installations (e.g., lines, tubes, pacemaker, NG tube, central venous catheter, prostheses, hardware, clips, stents).
- Do NOT include normal anatomy statements.

OUTPUT RULES:
- "original_row_index" MUST exactly match the index provided in the DATA.
- "findings" MUST be a JSON list of strings (each string is one bullet point; do NOT include bullet characters like "-" or "•" inside the string).
- If there are NO included findings (completely normal study), return an empty list: "findings": []
- Output JSON only. No extra text.

DATA:
{batch_data}
""".strip()

    raw_content = ""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You extract radiology findings. Output ONLY valid JSON."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
        )

        raw_content = response.choices[0].message.content or ""
        parsed = extract_json(raw_content)

        batch_results = parsed.get("results", [])
        if not isinstance(batch_results, list):
            raise ValueError("Model output JSON did not contain a 'results' list.")

        # Process results and append immediately
        for item in batch_results:
            idx = int(item["original_row_index"])
            findings_list = normalize_findings(item.get("findings", []))

            # Grab the current row from the source dataframe to keep all original data columns
            row_to_append = batch_df[batch_df['original_row_index'] == idx].copy()
            
            # Map decisions
            row_to_append["is_abnormal"] = 1 if findings_list else 0
            row_to_append["findings"] = json.dumps(findings_list, ensure_ascii=False)

            # Append the row to the existing collection
            df_existing = pd.concat([df_existing, row_to_append], ignore_index=True)

        end = min(start + BATCH_SIZE, len(df_source))
        print(f"Successfully processed rows {start} to {end}")

        # Drop duplicates based on tracking index just to safeguard file consistency before writing
        df_existing = df_existing.drop_duplicates(subset=['original_row_index'], keep='last')
        
        # Save real-time changes back to disk row-by-row
        df_existing.to_excel(OUTPUT_XLSX_PATH, index=False, engine="openpyxl")

    except Exception as e:
        print(f"\nError at batch starting row {start}: {e}")
        if raw_content:
            print("---- Raw model output (first 1500 chars) ----")
            print(raw_content[:1500])
            print("---- End raw output ----\n")
        time.sleep(2)

print(f"\nProcessing complete. All results saved safely to {OUTPUT_XLSX_PATH}")

No existing output file found. Initializing a new run.
Total reports to process: 423. Remaining: 423
Successfully processed rows 0 to 1
Successfully processed rows 1 to 2
Successfully processed rows 2 to 3
Successfully processed rows 3 to 4
Successfully processed rows 4 to 5
Successfully processed rows 5 to 6
Successfully processed rows 6 to 7
Successfully processed rows 7 to 8
Successfully processed rows 8 to 9
Successfully processed rows 9 to 10
Successfully processed rows 10 to 11
Successfully processed rows 11 to 12
Successfully processed rows 12 to 13
Successfully processed rows 13 to 14
Successfully processed rows 14 to 15
Successfully processed rows 15 to 16
Successfully processed rows 16 to 17
Successfully processed rows 17 to 18
Successfully processed rows 18 to 19
Successfully processed rows 19 to 20
Successfully processed rows 20 to 21
Successfully processed rows 21 to 22
Successfully processed rows 22 to 23
Successfully processed rows 23 to 24
Successfully processed rows 24